# 03 — Summarization Evaluation with ROUGE

ROUGE compares generated summaries with reference summaries.

This notebook calculates ROUGE-1, ROUGE-2 and ROUGE-L on a small validation sample.

In [1]:
from datasets import load_dataset
from rouge_score import rouge_scorer
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import pandas as pd

In [ ]:
MODEL_NAME = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

dataset = load_dataset("cnn_dailymail", "3.0.0", split="validation[:20]")

In [4]:
def generate_summary(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        ids = model.generate(
            **inputs,
            max_length=150,
            min_length=35,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True,
            no_repeat_ngram_size=3
        )

    return tokenizer.decode(ids[0], skip_special_tokens=True)

In [ ]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

results = []

for row in dataset:
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=True
    )

    results = []

    for row in dataset:
        generated = generate_summary(row["article"])
        scores = scorer.score(row["highlights"], generated)
        torch.cuda.empty_cache()  # Clear GPU cache if using CUDA

        results.append({
            "rouge1_f1": scores["rouge1"].fmeasure,
            "rouge2_f1": scores["rouge2"].fmeasure,
            "rougeL_f1": scores["rougeL"].fmeasure
        })

    scores_df = pd.DataFrame(results)
    scores_df.mean()
    torch.cuda.empty_cache()  # Clear GPU cache if using CUDA

    results.append({
        "rouge1_f1": scores["rouge1"].fmeasure,
        "rouge2_f1": scores["rouge2"].fmeasure,
        "rougeL_f1": scores["rougeL"].fmeasure
    })

scores_df = pd.DataFrame(results)
scores_df.mean()

In [ ]:
example = dataset[0]
generated = generate_summary(example["article"])

print("REFERENCE:")
print(example["highlights"])
print("\nGENERATED:")
print(generated)

## ROUGE Interpretation

- **ROUGE-1:** unigram overlap.
- **ROUGE-2:** bigram overlap.
- **ROUGE-L:** longest common subsequence.

Higher scores generally indicate greater overlap with the reference summary, but ROUGE is not a perfect measure of factual correctness or readability.

### Practical limitation

Abstractive models can hallucinate information. A production system should consider human review, factuality checks, domain testing and monitoring.